Loading the libraries

In [2]:
import sys
import os
import pandas as pd

# Add scripts directory to path
project_root = os.path.abspath('..')  # Go up to project root
scripts_path = os.path.join(project_root, 'scripts')
sys.path.insert(0, scripts_path)

# Now import
from utils import load_data, haversine, get_spatial_bounds

# Load data
df = load_data()

Loading from /Users/yuriirrgang/Library/Mobile Documents/com~apple~CloudDocs/Hertie/project-chaggg/data/cleaned/chicago_crimes_cleaned.parquet...


In [7]:
df.head()


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,updated_on,x_coordinate,y_coordinate,latitude,longitude,time,month,day,day_of_week,hour
0,1969564,HH162926,2002-01-01,063XX N WASHTENAW AV,0820,THEFT,$500 AND UNDER,STREET,False,False,...,2018-02-28T15:56:25.000,1157181.0,1941908.0,41.996366,-87.697157,00:00:00,1,1,1,0
1,1921971,HH105505,2002-01-01,085XX S CRANDON AV,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,False,False,...,2018-02-28T15:56:25.000,1192988.0,1848862.0,41.740240,-87.568491,00:00:00,1,1,1,0
2,1989110,HH188541,2002-01-01,047XX N HERMITAGE AV,0820,THEFT,$500 AND UNDER,RESIDENCE,False,False,...,2018-02-28T15:56:25.000,1163946.0,1931450.0,41.967528,-87.672569,00:00:00,1,1,1,0
3,1917950,HH101092,2002-01-01,028XX N PARKSIDE AV,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,...,2018-02-28T15:56:25.000,1138221.0,1918236.0,41.931773,-87.767479,00:00:00,1,1,1,0
4,1950239,HH139701,2002-01-01,066XX S KEDVALE AV,0842,THEFT,AGG: FINANCIAL ID THEFT,RESIDENCE,False,False,...,2018-02-28T15:56:25.000,1149846.0,1860270.0,41.772489,-87.726264,00:00:00,1,1,1,0


In [3]:


spatial_info = get_spatial_bounds(df)

print(f"Max distance in dataset: {spatial_info['max_distance']:.2f} km")
print(f"\nBounding box:")
print(f"  Latitude:  {spatial_info['bounds']['lat'][0]:.4f} to {spatial_info['bounds']['lat'][1]:.4f}")
print(f"  Longitude: {spatial_info['bounds']['lon'][0]:.4f} to {spatial_info['bounds']['lon'][1]:.4f}")

Max distance in dataset: 54.34 km

Bounding box:
  Latitude:  41.6446 to 42.0229
  Longitude: -87.9397 to -87.5245


In [8]:
validate_coordinates(df)
# One-liner approach
df_clean = df[
    (df['latitude'].between(41.624851, 42.07436)) & 
    (df['longitude'].between(-87.968437, -87.397217))
].copy()


Coordinate validation:
------------------------------------------------------------
Valid Chicago latitude range:  (41.624851, 42.07436)
Valid Chicago longitude range: (-87.968437, -87.397217)

Invalid latitude:  136 total
  - Null values:   0
  - Out of range:  136

Invalid longitude: 136 total
  - Null values:   0
  - Out of range:  136

Note: Not dropping these observations yet.
      Further analysis recommended before deciding.


In [3]:
import math

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth.
    
    Parameters:
        lat1, lon1: Latitude and longitude of point 1 (in decimal degrees)
        lat2, lon2: Latitude and longitude of point 2 (in decimal degrees)
    
    Returns:
        Distance in kilometers
    """

    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))#atan treats edge cases better on sphere
    R = 6371  # Earth's radius in kilometers
    return R * c

In [4]:
def cyclical_distance(value1, max_value, value2):
    """
    Calculate distance between two cyclical values using sine-cosine encoding.
    
    Parameters:
        value1, value2: The cyclical values (e.g., hour=23, hour=1)
        max_value: Maximum value in the cycle (e.g., 24 for hours, 7 for day_of_week)
    
    Returns:
        Distance between 0 and 1
    """
    # Convert to angles on unit circle
    angle1 = 2 * math.pi * value1 / max_value
    angle2 = 2 * math.pi * value2 / max_value
    
    # Compute sin/cos coordinates
    sin1, cos1 = math.sin(angle1), math.cos(angle1)
    sin2, cos2 = math.sin(angle2), math.cos(angle2)
    
    # Euclidean distance on unit circle
    distance = math.sqrt((sin1 - sin2)**2 + (cos1 - cos2)**2)
    
    # Normalize to [0, 1] (max distance on unit circle is 2)
    return distance / 2.0

In [6]:
def temporal_distance(month1, day1, hour1, dow1, month2, day2, hour2, dow2):
    """
    Calculate normalized temporal distance between two crimes.
    Uses cyclical encoding for month, hour, and day_of_week.
    
    Parameters:
        month1, day1, hour1, dow1: Time components of crime 1
        month2, day2, hour2, dow2: Time components of crime 2
        (month: 1-12, dow = day_of_week: 0=Monday, 6=Sunday)
    
    Returns:
        Temporal distance score between 0 and 1
    """
    # Cyclical features
    month_dist = cyclical_distance(month1, 12, month2)     # months range 1-12
    hour_dist = cyclical_distance(hour1, 24, hour2)        # hours range 0-23
    dow_dist = cyclical_distance(dow1, 7, dow2)            # day_of_week range 0-6
    
    # Linear feature (day of month is NOT cyclical)
    day_diff = abs(day1 - day2) / 31                       # days range 1-31

    # Average the four normalized differences
    return (month_dist + day_diff + hour_dist + dow_dist) / 4

In [7]:
def combined_distance(crime1, crime2, alpha=0.5, beta=0.5):
    """
    Combined spatial and temporal distance between two crimes.
    
    Parameters:
        crime1, crime2: dicts or rows with keys 'latitude', 'longitude', 
                        'month', 'day', 'hour', 'day_of_week'
        alpha: weight for spatial distance
        beta: weight for temporal distance
    
    Returns:
        Combined distance score
    """
    spatial = haversine(crime1['latitude'], crime1['longitude'], 
                        crime2['latitude'], crime2['longitude'])
    temporal = temporal_distance(crime1['month'], crime1['day'], crime1['hour'], crime1['day_of_week'],
                                 crime2['month'], crime2['day'], crime2['hour'], crime2['day_of_week'])
    
    # Normalize spatial to [0, 1] scale (Chicago is ~50km max distance)
    spatial_normalized = spatial / 50.0
    
    return alpha * spatial_normalized + beta * temporal